# Watermark Detection

## 1. Objective

We want to determine whether a generated text contains a watermark.

We compare two groups:
- 600 watermarked responses
- 600 unwatermarked responses

The detector uses a statistical score to identify the watermark.

Our goal is to measure how reliably watermarked texts can be
distinguished from unwatermarked texts.

## 2. Load Generated Texts

We use the two cleaned JSONL files created in the previous notebook.

First, we inspect the available fields to identify whether
detection scores were already computed during generation.

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()

if not (ROOT / "results").is_dir():
    ROOT = ROOT.parent

DATA = ROOT / "results" / "Cleaned Data"

def load_jsonl(filename):
    with (DATA / filename).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

wm = load_jsonl("watermarked_600.jsonl")
no_wm = load_jsonl("unwatermarked_600.jsonl")

print("Watermarked:", len(wm))
print("Unwatermarked:", len(no_wm))

print("\nWatermarked fields:")
print(list(wm[0].keys()))

print("\nUnwatermarked fields:")
print(list(no_wm[0].keys()))

Watermarked: 600
Unwatermarked: 600

Watermarked fields:
['prompt', 'completion', 'dataset_name', 'ages', 'length', 'z_score', 'p_value', 'token_color', 'mask', 'delta', 'convolution_kernel', 'topk', 'watermark_type', 'seeding_scheme', 'greenlist_type', 'greenlist_params', 'enforce_kl', 'n_iter', 'model_name', 'steps', 'gen_length', 'temperature', 'remasking', 'cfg_scale', 'mask_id', 'block_length']

Unwatermarked fields:
['prompt', 'completion', 'dataset_name', 'ages', 'length', 'watermark_type', 'model_name', 'steps', 'gen_length', 'temperature', 'remasking', 'cfg_scale', 'mask_id', 'block_length']


In [2]:
for name, rows in [("Watermarked", wm), ("Unwatermarked", no_wm)]:
    print(f"\n{name}")

    for field in ["z_score", "p_value"]:
        values = [row.get(field) for row in rows]
        available = sum(value is not None for value in values)

        print(f"{field}: {available}/{len(rows)} available")
        print("Example:", values[0])


Watermarked
z_score: 600/600 available
Example: 0.48031496062992124
p_value: 600/600 available
Example: 2.1016459654903982e-15

Unwatermarked
z_score: 0/600 available
Example: None
p_value: 0/600 available
Example: None


### Generation-Time Detection Scores

Detection scores were recorded during generation only for
watermarked responses.

The field `z_score` represents the observed proportion of green
tokens among scored tokens, rather than a standardized statistical
z-score.

To compare both experimental conditions fairly, we subsequently
apply the same detector to all saved completions.

## 3. Generation-Time Watermark Detection

The official detector uses a one-sided binomial test to compute
a p-value for each response.

We classify a response as watermarked when its p-value is
below 0.01 (1% significance level).

In this section, we analyze the detection scores recorded
during the original generation process.

These scores are available only for watermarked responses.

The results serve as a reference for comparison with the
unified detection procedure applied later to both conditions.

In [3]:
ALPHA = 0.01

detected = [
    row["p_value"] < ALPHA
    for row in wm
]

n_detected = sum(detected)
detection_rate = n_detected / len(wm) * 100

print(f"Watermarks detected: {n_detected}/{len(wm)}")
print(f"Detection rate: {detection_rate:.2f}%")

print("\nExample:")
print("Green token proportion:", wm[0]["z_score"])
print("P-value:", wm[0]["p_value"])
print("Watermark detected:", detected[0])

Watermarks detected: 576/600
Detection rate: 96.00%

Example:
Green token proportion: 0.48031496062992124
P-value: 2.1016459654903982e-15
Watermark detected: True


### Detection Rates by Dataset

We report generation-time detection rates separately for each
WaterBench dataset to examine differences across text domains.

These results are based on the original detection scores.
Unified detection results are reported later.

In [4]:
from collections import defaultdict

results = defaultdict(list)

for row in wm:
    dataset = row["dataset_name"].split("_batch_")[0]
    results[dataset].append(row["p_value"] < ALPHA)

for dataset, values in results.items():
    rate = 100 * sum(values) / len(values)
    print(f"{dataset}: {sum(values)}/{len(values)} detected ({rate:.2f}%)")

2-1_longform_qa: 198/200 detected (99.00%)
2-2_finance_qa: 195/200 detected (97.50%)
5-1_alpacafarm: 183/200 detected (91.50%)


## 4. Impact of Response Length on Watermark Detection

Watermark detection may be less reliable for shorter texts because fewer tokens are available for statistical analysis.

We observed a lower detection rate for AlpacaFarm (91.5%) compared to Longform QA (99.0%) and Finance QA (97.5%).

To investigate this difference, we compare the average response length across the three datasets.

In [ ]:
from collections import defaultdict

lengths = defaultdict(list)

for row in wm:
    dataset = row["dataset_name"].split("_batch_")[0]
    lengths[dataset].append(row["length"])

for dataset, values in lengths.items():
    print(f"{dataset}: {sum(values)/len(values):.1f} tokens on average")

2-1_longform_qa: 250.2 tokens en moyenne
2-2_finance_qa: 241.3 tokens en moyenne
5-1_alpacafarm: 246.1 tokens en moyenne


### Interpretation

Average response lengths are relatively similar across the three
datasets, ranging from 241.3 to 250.2 tokens.

Therefore, differences in average response length alone do not
explain the lower generation-time detection rate observed for AlpacaFarm.

A more detailed analysis would be required to determine whether
response length or other dataset-specific characteristics affect
watermark detectability.

## 5. False Positive Detection

A false positive occurs when the detector identifies a watermark
in a text that was generated without one.

We apply the original watermark detector to unwatermarked responses
using the same watermark key and parameters.

We first verify the detector on a small sample of 10
unwatermarked responses before evaluating the full dataset.

In [7]:
import sys
import subprocess
from pathlib import Path

REPO = ROOT / "diffusion-lm-watermark"

# Download the official code if necessary
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/eth-sri/diffusion-lm-watermark.git",
            str(REPO)
        ],
        check=True
    )

sys.path.insert(0, str(REPO / "src"))

from transformers import AutoTokenizer
from dlm_watermark.watermarks.diffusion_watermark import OurWatermark

# Only download the tokenizer, not the 8B model
tokenizer = AutoTokenizer.from_pretrained(
    "GSAI-ML/LLaDA-8B-Instruct",
    trust_remote_code=True
)

detector = OurWatermark(
    tokenizer=tokenizer,
    device="cpu",
    delta=4.0,
    convolution_kernel=[-1],
    topk=50,
    seeding_scheme="sumhash",
    greenlist_type="bernoulli",
    greenlist_params={"gamma": 0.25},
    enforce_kl=False,
    n_iter=1
)

print("Detector loaded successfully.")

/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] A new version of the following files was downloaded from https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct:
- configuration_llada.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Detector loaded successfully.


In [8]:
import torch

ALPHA = 0.01

for i, row in enumerate(no_wm[:10], 1):
    tokens = tokenizer(
        row["completion"],
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    with torch.no_grad():
        result = detector.detect(tokens)

    p_value = result["p_value"]
    detected = p_value < ALPHA

    print(
        f"Text {i:02d} | "
        f"p-value: {p_value:.6g} | "
        f"Detected: {detected}"
    )

Text 01 | p-value: 0.190483 | Detected: False
Text 02 | p-value: 0.576182 | Detected: False
Text 03 | p-value: 0.390633 | Detected: False
Text 04 | p-value: 0.756296 | Detected: False
Text 05 | p-value: 0.203045 | Detected: False
Text 06 | p-value: 0.918024 | Detected: False
Text 07 | p-value: 0.972624 | Detected: False
Text 08 | p-value: 0.863923 | Detected: False
Text 09 | p-value: 0.396526 | Detected: False
Text 10 | p-value: 0.568175 | Detected: False


### Preliminary Detection Check

All 10 unwatermarked responses are correctly classified as
unwatermarked at the 0.01 significance threshold.

Since the saved responses contain text rather than the original
token IDs, each completion is tokenized again before detection.

This preliminary check confirms that the detection pipeline
runs successfully, but does not establish its overall false
positive rate.

## 6. Detection Consistency Check

During generation, the watermark detector evaluates the original
token sequence produced by LLaDA.

However, our saved JSONL files contain decoded text rather than
the original generated token IDs.

To evaluate saved responses, we tokenize their completions again
using the LLaDA tokenizer. This process may produce a different
token sequence and therefore different detection p-values.

We first compare the stored and locally recomputed p-values
for 10 watermarked responses.

This preliminary check investigates the effect of retokenization
on detection consistency.

In [12]:
ALPHA = 0.01

for i, row in enumerate(wm[:10], 1):
    tokens = tokenizer(
        row["completion"],
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    result = detector.detect(tokens)

    stored = float(row["p_value"])
    local = float(result["p_value"])

    same_decision = (stored < ALPHA) == (local < ALPHA)

    print(
        f"Text {i:02d} | "
        f"Stored: {stored:.3g} | "
        f"Local: {local:.3g} | "
        f"Same decision: {same_decision}"
    )

Text 01 | Stored: 2.1e-15 | Local: 2.81e-15 | Same decision: True
Text 02 | Stored: 4.2e-09 | Local: 2.43e-09 | Same decision: True
Text 03 | Stored: 3.22e-14 | Local: 1.59e-14 | Same decision: True
Text 04 | Stored: 3.47e-16 | Local: 1.56e-16 | Same decision: True
Text 05 | Stored: 4.89e-08 | Local: 2.1e-08 | Same decision: True
Text 06 | Stored: 9.98e-07 | Local: 4.24e-07 | Same decision: True
Text 07 | Stored: 1.1e-16 | Local: 4.74e-17 | Same decision: True
Text 08 | Stored: 2.14e-05 | Local: 2.7e-05 | Same decision: True
Text 09 | Stored: 2.7e-07 | Local: 1.72e-07 | Same decision: True
Text 10 | Stored: 2.48e-14 | Local: 3.32e-14 | Same decision: True


## 7. Generation-Time vs Local Detection

We compare generation-time and locally recomputed detection
decisions for all 600 watermarked responses.

The local detector operates on retokenized saved completions,
whereas the generation-time detector used the original token IDs.

At a significance threshold of 0.01, 593 out of 600 decisions
agree. The remaining 7 responses were initially classified as
unwatermarked but are detected after local retokenization.

These differences are consistent with changes in token
representation, although other implementation differences
cannot be completely excluded.

In [13]:
ALPHA = 0.01
matches = 0
differences = []

for i, row in enumerate(wm, 1):
    tokens = tokenizer(
        row["completion"],
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    result = detector.detect(tokens)

    stored = float(row["p_value"])
    local = float(result["p_value"])

    if (stored < ALPHA) == (local < ALPHA):
        matches += 1
    else:
        differences.append({
            "index": i,
            "dataset": row["dataset_name"],
            "stored": stored,
            "local": local
        })

    if i % 100 == 0:
        print(f"Processed {i}/600")

print(f"\nMatching decisions: {matches}/600")
print(f"Different decisions: {len(differences)}")

for diff in differences[:10]:
    print(diff)

Processed 100/600
Processed 200/600
Processed 300/600
Processed 400/600
Processed 500/600
Processed 600/600

Matching decisions: 593/600
Different decisions: 7
{'index': 158, 'dataset': '2-1_longform_qa_batch_08.jsonl', 'stored': 0.014284366708780569, 'local': 0.009364029142329758}
{'index': 226, 'dataset': '2-2_finance_qa_batch_02.jsonl', 'stored': 0.015418810180054197, 'local': 0.008368141326488299}
{'index': 268, 'dataset': '2-2_finance_qa_batch_04.jsonl', 'stored': 0.013842074109066984, 'local': 0.007651634246283035}
{'index': 371, 'dataset': '2-2_finance_qa_batch_09.jsonl', 'stored': 0.01400599949828711, 'local': 0.007239519891685223}
{'index': 423, 'dataset': '5-1_alpacafarm_batch_02.jsonl', 'stored': 0.014702214279598468, 'local': 0.0074944089943219305}
{'index': 519, 'dataset': '5-1_alpacafarm_batch_06.jsonl', 'stored': 0.015475898135749634, 'local': 0.009406965929982743}
{'index': 562, 'dataset': '5-1_alpacafarm_batch_09.jsonl', 'stored': 0.01571441094072076, 'local': 0.008939

## 8. Investigating Detection Differences

Generation-time detection operates on the original generated
token IDs, while local detection retokenizes the saved text.

We investigate two representative cases:
- Text 1: matching detection decisions.
- Text 158: different detection decisions.

For each case, we compare the token counts, the number of
tokens included in the statistical test, and the resulting
p-values.

We also reconstruct the generation-time p-values using
the stored token colors and detection masks.

This helps validate the recorded detection statistics and
illustrates the effect of token representation differences.

In [14]:
from scipy.stats import binomtest

# Compare one matching case and one differing case
for index in [0, 157]:
    row = wm[index]

    # Original detection statistics
    colors = row["token_color"]
    mask = row["mask"]

    original_tokens = len(colors) + 1
    original_scored = sum(mask)
    original_green = sum(
        int(color) for color, keep in zip(colors, mask) if keep
    )

    # Verify the stored p-value
    checked_p = binomtest(
        original_green, original_scored, 0.25,
        alternative="greater"
    ).pvalue

    # Local detection statistics
    tokens = tokenizer(
        row["completion"],
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    local = detector.detect(tokens)

    print(f"\nText {index + 1}")
    print(f"Original tokens: {original_tokens}")
    print(f"Retokenized tokens: {tokens.shape[1]}")
    print(f"Original scored tokens: {original_scored}")
    print(f"Local scored tokens: {sum(local['mask'])}")
    print(f"Stored p-value: {row['p_value']:.6g}")
    print(f"Verified p-value: {checked_p:.6g}")
    print(f"Local p-value: {local['p_value']:.6g}")


Text 1
Original tokens: 300
Retokenized tokens: 298
Original scored tokens: 254
Local scored tokens: 252
Stored p-value: 2.10165e-15
Verified p-value: 2.10165e-15
Local p-value: 2.80966e-15

Text 158
Original tokens: 300
Retokenized tokens: 160
Original scored tokens: 140
Local scored tokens: 137
Stored p-value: 0.0142844
Verified p-value: 0.0142844
Local p-value: 0.00936403


### Interpretation

The stored p-values are successfully reproduced using the original
detection statistics.

Retokenization changes the token sequences and can affect detection
p-values. Across all 600 watermarked responses, 593 detection decisions
remain unchanged (98.83%), while 7 differ.

For subsequent robustness experiments, we consistently apply
the local detector to both original and attacked texts.

## 9. Unified Watermark Detection

Previous experiments revealed differences between detection on
the original generated tokens and detection on retokenized text.

To ensure a fair comparison, we now apply the same local detector
to both watermarked and unwatermarked responses.

We use:
- The same tokenizer and watermark configuration
- The same detection function
- The same significance threshold (0.01)

These results will serve as our baseline for future removal attacks.

In [15]:
import torch

ALPHA = 0.01

def detect_watermark(text):
    tokens = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    with torch.no_grad():
        result = detector.detect(tokens)

    p_value = float(result["p_value"])

    return {
        "p_value": p_value,
        "detected": p_value < ALPHA
    }


# Detect both groups using the same method
wm_results = [detect_watermark(row["completion"]) for row in wm]
no_wm_results = [detect_watermark(row["completion"]) for row in no_wm]

print("Detection completed for all 1,200 responses.")

Detection completed for all 1,200 responses.


In [16]:
TP = sum(r["detected"] for r in wm_results)
FN = len(wm_results) - TP

FP = sum(r["detected"] for r in no_wm_results)
TN = len(no_wm_results) - FP

TPR = TP / (TP + FN)
FPR = FP / (FP + TN)
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("Unified Watermark Detection Results")
print("-----------------------------------")
print(f"True Positives:  {TP}")
print(f"False Negatives: {FN}")
print(f"False Positives: {FP}")
print(f"True Negatives:  {TN}")

print(f"\nTPR: {TPR:.2%}")
print(f"FPR: {FPR:.2%}")
print(f"Accuracy: {accuracy:.2%}")

Unified Watermark Detection Results
-----------------------------------
True Positives:  583
False Negatives: 17
False Positives: 1
True Negatives:  599

TPR: 97.17%
FPR: 0.17%
Accuracy: 98.50%


## 10. Conclusion — Watermark Detection Results

We evaluated 1,200 responses generated by LLaDA-8B-Instruct using the same local watermark detector, tokenizer, and significance threshold ($p < 0.01$).

### Detection Results

| Metric | Watermarked (600) | Unwatermarked (600) |
|---|---:|---:|
| Watermark detected | 583 (97.17%) | 1 (0.17%) |
| Watermark not detected | 17 (2.83%) | 599 (99.83%) |

### Overall Performance

| Metric | Score |
|---|---:|
| True Positive Rate (TPR) | 97.17% |
| False Positive Rate (FPR) | 0.17% |
| Accuracy | 98.50% |
| Detection threshold | p < 0.01 |

### Interpretation

The detector successfully identifies **97.17% of watermarked responses**, while incorrectly classifying only **0.17% of unwatermarked responses** as watermarked.

These results demonstrate that the watermark is highly detectable under our experimental conditions, with very few false positives.

Unlike the initial generation-time evaluation, both groups were evaluated using the same local detection procedure.

This establishes a consistent baseline for future watermark removal attacks.

These results characterize watermark detectability on
unmodified generated responses, before any adversarial attack.

## 11. Export Detection Results

We save the detection scores for all 1,200 responses.

These results will be used as a baseline for evaluating
watermark removal attacks without repeating the detection process.

In [17]:
import json

OUTPUT = ROOT / "results" / "detection_results.jsonl"

with OUTPUT.open("w", encoding="utf-8") as f:
    for condition, rows, scores in [
        ("watermarked", wm, wm_results),
        ("unwatermarked", no_wm, no_wm_results)
    ]:
        for row, score in zip(rows, scores):
            result = {
                "dataset": row["dataset_name"].split("_batch_")[0],
                "condition": condition,
                "prompt": row["prompt"],
                "completion": row["completion"],
                "p_value": score["p_value"],
                "detected": score["detected"]
            }

            f.write(json.dumps(result, ensure_ascii=False) + "\n")

print("Saved:", OUTPUT)
print("Total responses:", len(wm_results) + len(no_wm_results))

Saved: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/detection_results.jsonl
Total responses: 1200
